In [ ]:
import os
import sys

for root, dirs, files in os.walk(os.getcwd()):
    if "pi0-analysis" in root.split("/"):
        pypath = root.split("pi0-analysis")[0] + "/pi0-analysis/analysis"
        break

sys.path.insert(0, pypath)

from python.analysis.NotebookUtils import init_notebook
%init_notebook

import pickle
import pandas as pd
from rich import print

from utils import select_mc_events, extract_pfo_data

In [ ]:
import awkward as ak
from python.analysis import Master

def _fsi_topology(n_pipm, n_pi0):
    """4 FSI topologies (Bhuller Table 4.1), charged pions merged (no B-field)."""
    if n_pipm == 0 and n_pi0 == 0: return 0   # absorption
    if n_pipm == 0 and n_pi0 == 1: return 1   # charge exchange
    if n_pipm == 1 and n_pi0 == 0: return 2   # single-pion production
    return 3                                   # pion production (multi)
TOPO_NAMES = {0:"absorption", 1:"charge_exchange", 2:"single_pi", 3:"pion_production", -1:"non_signal"}

def extract_event_data(selected_data, verbose=False):
    """One row per event, keyed by the SAME global event_num extract_pfo_data uses."""
    if isinstance(selected_data, Master.Data):
        data_list = [selected_data]
    elif isinstance(selected_data, list) and selected_data and isinstance(selected_data[0], tuple):
        data_list = [d for (_, d) in selected_data]
    else:
        data_list = selected_data

    events = []
    event_num = -1
    for mc in data_list:
        has_truth = ak.count(mc.trueParticlesBT.pdg) > 0
        tp = mc.trueParticles
        for event in range(len(mc.recoParticles.track_chi2_proton)):
            event_num += 1
            row = {"event_number": event_num,
                   "n_pfos": int(len(mc.recoParticles.track_chi2_proton[event]))}
            if has_truth:
                n_pipm = int(tp.nPiPlus[event]) + int(tp.nPiMinus[event])
                n_pi0  = int(tp.nPi0[event])
                end    = str(tp.true_beam_endProcess[event])
                sig    = (end == "pi+Inelastic")
                topo   = _fsi_topology(n_pipm, n_pi0) if sig else -1
                row.update({
                    "beam_KE_front_face": float(tp.beam_KE_front_face[event]),
                    "true_beam_endProcess": end,
                    "nPiPlus": int(tp.nPiPlus[event]), "nPiMinus": int(tp.nPiMinus[event]),
                    "nPi0": n_pi0, "nProton": int(tp.nProton[event]),
                    "true_topology": topo, "true_topology_name": TOPO_NAMES[topo],
                })
            events.append(row)

    if verbose:
        from collections import Counter
        print(f"events extracted: {len(events):,}")
        if events and "true_topology_name" in events[0]:
            print(Counter(e["true_topology_name"] for e in events))
    return events

In [ ]:
mc_files = [
    "/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set0.root",
    "/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set1.root",
    "/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set2.root",
    "/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set3.root",
    "/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_reco1_sce_datadriven_v1_ntuple_v09_41_00_03.root",
]

In [ ]:
# Apply fiducial + beam selection and get event counts
selected_data, selection_stats = select_mc_events(mc_files, verbose=True)

In [ ]:
pfo_data, pfo_stats = extract_pfo_data(selected_data, max_sequence_length=222, verbose=True)
df = pd.DataFrame(pfo_data)

In [ ]:
# where to write the extracted data pkl (note: underscore folder name)
OUTPUT_PATH = "/home/pemb7173/PDSP-pion-classification/extracted_mc/mc_events.pkl"

event_data = extract_event_data(selected_data, verbose=True)
events_path = OUTPUT_PATH
with open(events_path, "wb") as f:
    pickle.dump(pd.DataFrame(event_data), f)
print(f"Saved {len(event_data):,} event rows to {events_path}")

In [ ]:
OUTPUT_DIR = os.path.dirname(OUTPUT_PATH)
if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(df, f)
print(f"Saved {df} to {OUTPUT_PATH}")